# Lab | Agent & Vector store: my solution

Guide (Spanish): `lab_agent_vector_store_guide.html`, step 8 and 9.

Plan: the speech and Ruff stores come from the demo, then I add a third store with Shakespeare's sonnets.

## 1. Setup (code cells 3–5 of the lab, adjusted)

Two adjustments: an IPv4 client (avoids very slow calls on this Mac) and `ChatOpenAI`, because `gpt-3.5-turbo-instruct` was retired by OpenAI.

In [1]:
from langchain.chains import RetrievalQA
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAI, OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader, WebBaseLoader
from langchain.agents import AgentType, Tool, initialize_agent
import httpx

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
import os
from dotenv import load_dotenv, find_dotenv

if not load_dotenv(find_dotenv()):
    load_dotenv(os.path.expanduser("~/Desktop/Ironhack/.env"), override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
print("API key loaded:", bool(OPENAI_API_KEY))

API key loaded: True


In [3]:
ipv4_client = httpx.Client(transport=httpx.HTTPTransport(local_address="0.0.0.0"))

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=OPENAI_API_KEY, http_client=ipv4_client)
embeddings = OpenAIEmbeddings(api_key=OPENAI_API_KEY, http_client=ipv4_client)

## 2. Store 1: state of the union speech

In [4]:
import time
t0 = time.time()

loader = TextLoader("./datasets/state_of_the_union.txt")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)

docsearch = Chroma.from_documents(texts, embeddings, collection_name="state-of-union")
state_of_union = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=docsearch.as_retriever()
)
print(f"{len(texts)} chunks in {time.time() - t0:.1f}s")

42 chunks in 1.6s


## 3. Store 2: Ruff FAQ

`WebBaseLoader` downloads the page with `requests`, not with `httpx`, so the IPv4 client above does not help it. The first lines force IPv4 for `requests` too. The timing prints show which part is slow: the download or the embeddings.

In [5]:
import socket
import urllib3.util.connection as urllib3_cn
urllib3_cn.allowed_gai_family = lambda: socket.AF_INET  # IPv4 only for requests

t0 = time.time()
loader = WebBaseLoader("https://beta.ruff.rs/docs/faq/", requests_kwargs={"timeout": 30})
docs = loader.load()
print(f"download: {time.time() - t0:.1f}s")

download: 0.4s


In [6]:
t0 = time.time()
ruff_texts = text_splitter.split_documents(docs)
ruff_db = Chroma.from_documents(ruff_texts, embeddings, collection_name="ruff")
ruff = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=ruff_db.as_retriever()
)
print(f"{len(ruff_texts)} chunks, embeddings + store: {time.time() - t0:.1f}s")

Created a chunk of size 2209, which is longer than the specified 1000
Created a chunk of size 3190, which is longer than the specified 1000
Created a chunk of size 1017, which is longer than the specified 1000
Created a chunk of size 2313, which is longer than the specified 1000


23 chunks, embeddings + store: 0.9s


---

## 🚀 Your turn: Shakespeare's sonnets

The file `datasets/sonnets.txt` is already downloaded. Fill the `____` gaps. Run each store cell **once** (re-running duplicates the data in Chroma; if that happens, restart the kernel and run everything from the top).

### 4. Build the sonnets store (task points 1 and 2)

In [7]:
doc_path = "./datasets/sonnets.txt"
loader = TextLoader(doc_path)
documents = loader.load()
sonnets_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
sonnets_texts = sonnets_splitter.split_documents(documents)

sonnets_db = Chroma.from_documents(sonnets_texts, embeddings, collection_name="sonnets")
sonnets = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=sonnets_db.as_retriever()
)


### 5. Write the tool (task point 3)

In [9]:
sonnets_tool = Tool(
    name="Shakespeare Sonnets QA System",
    func=sonnets.run,
    description="useful for when you need to answer questions about Shakespeare's sonnets (love, time, beauty, the poems themselves). Input should be a fully formed question.",
)

### 6. Rebuild the agent (task point 4)

In [11]:
ruff_tool = Tool(
    name="Ruff QA System",
    func=ruff.run,
    description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question.",
)

tools = [sonnets_tool, ruff_tool]

agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

/var/folders/5t/lmqvg1xx4nl0pnr06vf0vy2h0000gn/T/ipykernel_58937/2540638460.py:9: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 1.0. Use Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc. instead.
  agent = initialize_agent(


### 7. Put it to the test (task point 5)

Before running each question, write down which tool you expect.

In [12]:
# 1) only the sonnets can answer
agent.invoke("What does Shakespeare say about the passing of time in his sonnets?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


I need to gather information about Shakespeare's perspective on the passing of time as expressed in his sonnets. This will likely involve themes of mortality, beauty, and the effects of time on love and life. 
Action: Shakespeare Sonnets QA System
Action Input: What does Shakespeare say about the passing of time in his sonnets?
Observation: In Shakespeare's sonnets, he reflects on the passing of time as an inevitable force that leads to decay and the loss of beauty. He often portrays time as a relentless entity that affects all living things, causing youth and beauty to fade. For example, he describes how everything that grows holds perfection for only a brief moment and how time ultimately takes away the vibrancy of youth, leading to decay and death. Despite this, he expresses hope that his verses will immortalize the worth of the beloved, suggesting that art can defy time's cruel hand by preserving beauty in memory. Overall, Shakespeare's treatment of time is one of both lamentation 

{'input': 'What does Shakespeare say about the passing of time in his sonnets?',
 'output': "In Shakespeare's sonnets, he reflects on the passing of time as an inevitable force that leads to decay and the loss of beauty. He portrays time as a relentless entity that affects all living things, causing youth and beauty to fade. While lamenting its destructive power, he also expresses hope that his verses will immortalize the worth of the beloved, suggesting that art can defy time's cruel hand by preserving beauty in memory."}

In [13]:
# 2) only Ruff can answer
agent.invoke("Why use ruff over flake8?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


I need to compare the features and benefits of Ruff and Flake8 to provide a comprehensive answer about why one might choose Ruff over Flake8. 
Action: Ruff QA System
Action Input: Why should someone use Ruff over Flake8 for Python linting?
Observation: Someone might choose to use Ruff over Flake8 for Python linting for several reasons:

1. **Performance**: Ruff is designed to be faster than Flake8, providing quicker feedback on lint violations.

2. **Comprehensive Rule Set**: Ruff implements over 900 rules, including many from popular Flake8 plugins, which allows it to catch a wider range of issues.

3. **Automatic Fixing**: Ruff is capable of automatically fixing its own lint violations, which Flake8 does not support.

4. **Native Implementation**: Ruff implements all rules natively and does not rely on third-party plugins, simplifying the setup and configuration process.

5. **Compatibility with Black**: Ruff's linter is compatible with Black out-of-the-box, making it easier to integ

{'input': 'Why use ruff over flake8?',
 'output': 'Someone might choose to use Ruff over Flake8 for Python linting due to its performance, comprehensive rule set, automatic fixing capabilities, native implementation, compatibility with Black, ease of use, and streamlined setup. Ruff is designed to be faster and catch a wider range of issues, making it a modern and efficient alternative to Flake8.'}

In [14]:
# 3) needs both tools
agent.invoke("Why use ruff over flake8, and what does Shakespeare say about beauty fading in his sonnets?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


I need to address two separate topics: the advantages of using Ruff over Flake8 for Python linting, and Shakespeare's perspective on beauty fading as expressed in his sonnets. I will first gather information about Ruff and Flake8, then I will look into Shakespeare's sonnets regarding beauty.

Action: Ruff QA System  
Action Input: "Why use ruff over flake8?"  
Observation: There are several reasons to consider using Ruff over Flake8:

1. **Performance**: Ruff is designed to be faster than Flake8, providing quicker feedback on lint violations.

2. **Comprehensive Rule Set**: Ruff implements over 900 rules, including many from popular Flake8 plugins, while Flake8 has around 409 total rules. This means Ruff can catch a wider variety of issues.

3. **Automatic Fixing**: Ruff is capable of automatically fixing its own lint violations, which Flake8 does not support.

4. **Native Implementation**: Ruff implements all rules natively and does not rely on third-party plugins, which can simplify 

{'input': 'Why use ruff over flake8, and what does Shakespeare say about beauty fading in his sonnets?',
 'output': "Ruff is preferred over Flake8 for several reasons: it is faster, has a more comprehensive rule set with over 900 rules, can automatically fix lint violations, and is implemented natively without relying on third-party plugins. In contrast, Shakespeare's sonnets express that beauty is transient and fades over time, emphasizing that true beauty can be preserved through offspring and that societal standards of beauty are subject to change."}

### 8. Same test with `return_direct=True`

Create the two tools again with `return_direct=True`, build a new agent with them and repeat question 3.

In [15]:
sonnets_tool_direct = Tool(
    name="Shakespeare Sonnets QA System",
    func=sonnets.run,
    description="useful for when you need to answer questions about Shakespeare's sonnets (love, time, beauty, the poems themselves). Input should be a fully formed question.",
    return_direct=True,
)
ruff_tool_direct = Tool(
    name="Ruff QA System",
    func=ruff.run,
    description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question.",
    return_direct=True,
)

agent_direct = initialize_agent(
    [sonnets_tool_direct, ruff_tool_direct], llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True,
)
agent_direct.invoke("Why use ruff over flake8, and what does Shakespeare say about beauty fading in his sonnets?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


I need to address two separate topics: the advantages of using Ruff over Flake8 for Python linting, and Shakespeare's perspective on beauty fading as expressed in his sonnets. I will first gather information about Ruff and Flake8, then I will look into Shakespeare's sonnets regarding beauty.

Action: Ruff QA System  
Action Input: "Why use ruff over flake8?"  
Observation: There are several reasons to consider using Ruff over Flake8:

1. **Performance**: Ruff is designed to be faster than Flake8, providing quicker feedback on lint violations.

2. **Comprehensive Rule Set**: Ruff implements over 900 rules, including many from popular Flake8 plugins, while Flake8 has around 409 total rules. This means Ruff can catch a wider variety of issues.

3. **Automatic Fixing**: Ruff is capable of automatically fixing its own lint violations, which Flake8 does not support.

4. **No Custom Rules**: Ruff does not support custom lint rules, but it re-implements popular Flake8 plugins natively, which c

{'input': 'Why use ruff over flake8, and what does Shakespeare say about beauty fading in his sonnets?',
 'output': 'There are several reasons to consider using Ruff over Flake8:\n\n1. **Performance**: Ruff is designed to be faster than Flake8, providing quicker feedback on lint violations.\n\n2. **Comprehensive Rule Set**: Ruff implements over 900 rules, including many from popular Flake8 plugins, while Flake8 has around 409 total rules. This means Ruff can catch a wider variety of issues.\n\n3. **Automatic Fixing**: Ruff is capable of automatically fixing its own lint violations, which Flake8 does not support.\n\n4. **No Custom Rules**: Ruff does not support custom lint rules, but it re-implements popular Flake8 plugins natively, which can simplify the setup.\n\n5. **Compatibility with Formatters**: Ruff is designed to work well with formatters like Black, and it can be used alongside them without conflicts.\n\n6. **Drop-in Replacement**: Under certain conditions (without or with a s

## Reflection (task point 6)

- Only-sonnets question: the agent chose Shakespeare Sonnets QA System (correct)
- Only-Ruff question: the agent chose Ruff QA System (correct)
- Combined question: it used 2 tools (Ruff first, then the sonnets) and the final answer combined both: Ruff is faster, has 900+ rules and auto-fixes, and Shakespeare presents beauty as fleeting and preserved through offspring
- With `return_direct=True`: the agent called only the first tool (Ruff), returned its raw answer and stopped. The Shakespeare half of the question was never answered, because without a final reasoning step the agent cannot combine sources. `return_direct` is fine as a router for single-source questions, not for multi-hop ones.
- What I would change in the tool descriptions: make them more specific and mutually exclusive, e.g. "Shakespeare's 154 sonnets (poems about love, time, beauty)... NOT for programming questions" and "Ruff, a Python linter (flake8, Black, lint rules)... NOT for literature". I would also add "if the question has several parts, ask one fully formed question per tool" so multi-part questions get split correctly.